In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

path = '../../data/bbob/res_16'

# Randomly pick one generated tensor (.npz) under `path`
npz_files = []
for root, _, files in os.walk(path):
    for name in files:
        if name.endswith('.npz'):
            npz_files.append(os.path.join(root, name))

if not npz_files:
    raise FileNotFoundError(f"No .npz files found under: {path}")

chosen = random.choice(npz_files)
print("Chosen:", chosen)

with np.load(chosen) as data:
    plots = data["plots"]  # (k, r, r)
    mask = data["masks"]    # (k, r, r)
    stats = data["stats"]  # (k,)

n_show = min(9, plots.shape[0])

# Plots
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()
for i in range(9):
    ax = axes[i]
    ax.axis('off')
    if i < n_show:
        ax.imshow(plots[i], cmap='YlGnBu_r', interpolation='nearest')

        # Set stats in title
        stat = stats[i]
        ell, range_, iqr = stat
        
        ax.set_title(f"$\ell={ell:.2f}, r={range_:.2f}, iqr={iqr:.2f}$", fontsize=9)
plt.tight_layout()
plt.show()

# Corresponding masks
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()
for i in range(9):
    ax = axes[i]
    ax.axis('off')
    if i < n_show:
        ax.imshow(mask[i], cmap='gray', interpolation='nearest', vmin=0, vmax=1)
        # ax.set_title(f"mask #{i}", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

path = '../../data/bbob/res_16'

# Randomly pick one generated tensor (.npz) under `path`
npz_files = []
for root, _, files in os.walk(path):
    for name in files:
        if name.endswith('.npz'):
            npz_files.append(os.path.join(root, name))

if not npz_files:
    raise FileNotFoundError(f"No .npz files found under: {path}")

chosen = random.choice(npz_files)
print("Chosen:", chosen)

with np.load(chosen) as data:
    plots = data["plots"]  # (k, r, r)
    mask = data["masks"]    # (k, r, r)
    stats = data["stats"]  # (k,)

n_show = min(5, plots.shape[0])

# Plots
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.ravel()
for i in range(5):
    ax = axes[i]
    ax.axis('off')
    if i < n_show:
        ax.imshow(plots[i], cmap='YlGnBu_r', interpolation='nearest')

        # Set stats in title
        stat = stats[i]
        ell, range_, iqr = stat
        
        ax.set_title(f"$\ell={ell:.2f}$\n$\Delta={range_:.2f}$\n$q={iqr:.2f}$", fontsize=12, loc='left')
# plt.tight_layout()
# plt.show()

# Corresponding masks
# fig, axes = plt.subplots(1, 5, figsize=(12, 10))
# axes = axes.ravel()
for i in range(5):
    ax = axes[i+5]
    ax.axis('off')
    if i < n_show:
        ax.imshow(mask[i].astype(float) * 0.8 + 0.1, cmap='gray', interpolation='nearest', vmin=0, vmax=1)
        # ax.set_title(f"mask #{i}", fontsize=9)
plt.tight_layout()
# plt.show()
# Should save as svg
plt.savefig('plot_mask_check.svg', bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib as mpl
import cocoex

# Reuse the same sampler / normaliser as the dataset generator
from auxiliary_functions import sample_random_slices, normalise_plots_masked_inplace

# --------------------------- Plot styling (minimal) --------------------------- #
try:
    plt.style.use("seaborn-v0_8-white")
except Exception:
    pass

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.transparent": False,
    "savefig.facecolor": "white",
    "axes.titlepad": 6,
    "axes.titlesize": 10,
    "font.size": 9,
    "mathtext.fontset": "stix",
    "font.family": "DejaVu Serif",
})

# This cell samples oriented 2D square slices in [0,1]^d similarly to the generator:
#   - centres: Sobol in [0,1]^d
#   - side length ell: log-uniform in [ell_min, ell_max] via Sobol u
#   - orientation: approx Haar via Gaussian -> QR (d x 2 orthonormal frame)
# Then it evaluates a chosen BBOB problem on each slice grid and plots
# (1) a 3D view of the slice squares and their exact cube-clipped polygons,
# (2) the corresponding 2D evaluation plots and validity masks.

# ---- Parameters ----
num_squares = 6           # number of sampled slices (k_views)
dim = 3                   # must be >= 2; set to 3 for the 3D demo
resolution = 32         # r: pixels per side of each slice grid
ell_min = 0.02
ell_max = 0.7
seed = 111
eval_batch_size = 2000

# Choose a BBOB problem instance to evaluate (same idea as plot_generation_soo_extensive.py)
func_id = 17
instance_id = 1

# Output filenames
demo_svg = f"demo_3d_f{func_id}_i{instance_id}_seed{seed}.svg"
eval_prefix = f"eval_plots_masks_f{func_id}_i{instance_id}_seed{seed}"
colorbar_svg = f"eval_colorbar_f{func_id}_i{instance_id}_seed{seed}.svg"

# Physical evaluation bounds (matches generator)
bounds = np.array([[-5.0, 5.0]] * dim, dtype=np.float32)
lows, highs = bounds[:, 0], bounds[:, 1]
widths = highs - lows

# --------------------------- Exact cube clipping --------------------------- #
def _segment_plane_intersection(p0: np.ndarray, p1: np.ndarray, axis: int, value: float) -> np.ndarray:
    denom = float(p1[axis] - p0[axis])
    if abs(denom) < 1e-12:
        return p0.copy()
    t = (value - float(p0[axis])) / denom
    t = float(np.clip(t, 0.0, 1.0))
    return (p0 + t * (p1 - p0)).astype(np.float32, copy=False)


def _clip_polygon_with_halfspace(
    poly: list[np.ndarray], *, axis: int, value: float, keep_ge: bool
) -> list[np.ndarray]:
    if not poly:
        return []
    eps = 1e-9
    def inside(p: np.ndarray) -> bool:
        if keep_ge:
            return float(p[axis]) >= value - eps
        return float(p[axis]) <= value + eps

    out: list[np.ndarray] = []
    for i in range(len(poly)):
        curr = poly[i]
        prev = poly[i - 1]
        curr_in = inside(curr)
        prev_in = inside(prev)

        if curr_in:
            if not prev_in:
                out.append(_segment_plane_intersection(prev, curr, axis, value))
            out.append(curr)
        elif prev_in:
            out.append(_segment_plane_intersection(prev, curr, axis, value))
    return out


def clip_polygon_to_unit_cube(poly: np.ndarray) -> np.ndarray:
    """Clip a convex polygon (N,3) against [0,1]^3 using Sutherland–Hodgman."""
    pts: list[np.ndarray] = [np.asarray(p, dtype=np.float32) for p in poly]
    # x >= 0, x <= 1, y >= 0, y <= 1, z >= 0, z <= 1
    pts = _clip_polygon_with_halfspace(pts, axis=0, value=0.0, keep_ge=True)
    pts = _clip_polygon_with_halfspace(pts, axis=0, value=1.0, keep_ge=False)
    pts = _clip_polygon_with_halfspace(pts, axis=1, value=0.0, keep_ge=True)
    pts = _clip_polygon_with_halfspace(pts, axis=1, value=1.0, keep_ge=False)
    pts = _clip_polygon_with_halfspace(pts, axis=2, value=0.0, keep_ge=True)
    pts = _clip_polygon_with_halfspace(pts, axis=2, value=1.0, keep_ge=False)
    if not pts:
        return np.empty((0, 3), dtype=np.float32)
    return np.stack(pts, axis=0).astype(np.float32, copy=False)

# ---- Sampling (reuse generator logic) ----
batch = sample_random_slices(
    d=dim,
    k_views=num_squares,
    r=resolution,
    bounds=bounds,
    ell_min=ell_min,
    ell_max=ell_max,
    use_sobol=True,
    seed=seed,
 )

# batch.X is already in physical bounds and clipped; reshape for evaluation
X_phys = batch.X.transpose(0, 2, 3, 1)  # (k,r,r,d)
mask = batch.mask
ell = batch.ell
centers = batch.centers
B = batch.B

if centers is None or B is None:
    raise RuntimeError("Sampler did not provide centers/B (unexpected).")

# Also build 3D square polygons (corners) in unit cube coordinates
corner_offsets = np.array([[-0.5, -0.5], [-0.5, 0.5], [0.5, 0.5], [0.5, -0.5]], dtype=np.float32)
squares_raw = []
squares_clip = []
valid_flags = []
for i in range(num_squares):
    c = centers[i]          # (d,)
    b0, b1 = B[i, 0], B[i, 1]
    side = float(ell[i])

    corners_raw = np.array([c + side * (dx * b0 + dy * b1) for dx, dy in corner_offsets], dtype=np.float32)
    is_valid = bool(np.all((corners_raw >= 0.0) & (corners_raw <= 1.0)))
    valid_flags.append(is_valid)

    corners_clip = clip_polygon_to_unit_cube(corners_raw)
    squares_raw.append(corners_raw)
    squares_clip.append(corners_clip)

# ---- 3D demo plot (save to svg) ----
fig = plt.figure(figsize=(7.6, 6.4))
ax = fig.add_subplot(111, projection="3d")
try:
    ax.set_proj_type("persp")
except Exception:
    pass
ax.view_init(elev=18, azim=35)

# Make panes transparent for a cleaner look
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    try:
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor((0.2, 0.2, 0.2, 0.2))
    except Exception:
        pass
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

cube_vertices = np.array(
    [[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0], [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]],
    dtype=float,
 )
cube_edges = np.array(
    [[0, 1], [1, 2], [2, 3], [3, 0], [4, 5], [5, 6], [6, 7], [7, 4], [0, 4], [1, 5], [2, 6], [3, 7]],
    dtype=int,
 )
for a, b in cube_edges:
    s0, e0 = cube_vertices[a], cube_vertices[b]
    ax.plot3D(*zip(s0, e0), color="black", alpha=0.25, linewidth=0.9)

# Plot squares: invalid -> faint unclipped + solid clipped
for corners_raw, poly_clip, is_valid in zip(squares_raw, squares_clip, valid_flags):
    face = (0.15, 0.60, 0.25, 0.75) if is_valid else (0.80, 0.15, 0.20, 0.75)
    edge = (0, 0, 0, 0.65)

    if not is_valid:
        poly_raw = Poly3DCollection([corners_raw], alpha=0.10, linewidths=0.7, edgecolors=edge, antialiaseds=True)
        poly_raw.set_facecolor(face)
        ax.add_collection3d(poly_raw)

    if poly_clip.shape[0] >= 3:
        poly_solid = Poly3DCollection([poly_clip], alpha=0.75, linewidths=0.9, edgecolors=edge, antialiaseds=True)
        poly_solid.set_facecolor(face)
        ax.add_collection3d(poly_solid)

ax.set_title(rf"{len(squares_raw)} slices; solid = intersection with $[0,1]^3$")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.set_box_aspect([1, 1, 1])
fig.tight_layout(pad=0.2)
fig.savefig(demo_svg, bbox_inches="tight", pad_inches=0.03)
plt.show()

# ---- Evaluate a BBOB problem on the sampled slice grids ----
suite = cocoex.Suite(
    "bbob",
    "",
    f"dimensions:{dim} function_indices:{func_id} instance_indices:{instance_id}",
 )
problem = suite.get_problem(0)

points = X_phys.reshape(-1, dim)  # (k*r*r, d)
values = np.empty((points.shape[0],), dtype=np.float32)
for a in range(0, points.shape[0], eval_batch_size):
    b = min(a + eval_batch_size, points.shape[0])
    values[a:b] = np.array([problem(x) for x in points[a:b]], dtype=np.float32)

plots = values.reshape(num_squares, resolution, resolution)  # (k,r,r)

# Stats from pre-normalised values (masked)
valid = (mask.astype(bool)) & np.isfinite(plots)
stats = np.zeros((num_squares, 3), dtype=np.float32)
stats[:, 0] = ell.astype(np.float32, copy=False)
for i in range(num_squares):
    vi = valid[i]
    if not np.any(vi):
        continue
    vals = plots[i][vi]
    mn, mx = float(vals.min()), float(vals.max())
    stats[i, 1] = np.float32(mx - mn)
    q1, q3 = np.quantile(vals, [0.25, 0.75])
    stats[i, 2] = np.float32(q3 - q1)

# Mask-aware normalisation to [0,1] (same behavior as generator)
plots_norm = plots.astype(np.float32, copy=True)
normalise_plots_masked_inplace(plots_norm, mask)

# ---- Save a standalone colorbar (same cmap + [0,1] normalization) ----
cmap = "YlGnBu_r"
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0.0, vmax=1.0), cmap=cmap)
sm.set_array([])
fig_cb, ax_cb = plt.subplots(figsize=(0.5, 4.2))
cbar = fig_cb.colorbar(sm, cax=ax_cb)
cbar.outline.set_linewidth(0.6)
ax_cb.tick_params(length=2.0, width=0.6, labelsize=12)
fig_cb.tight_layout(pad=0.15)
fig_cb.savefig(colorbar_svg, bbox_inches="tight", pad_inches=0.03)
# plt.close(fig_cb)

# ---- Save evaluation plots and masks ONE BY ONE ----
n_show = num_squares
saved = []
for i in range(n_show):
    ell_i, delta_i, iqr_i = stats[i]
    out_svg = f"{eval_prefix}_{i+1}.svg"
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(3.4, 6.4),
        gridspec_kw={"height_ratios": [1.0, 1.0], "hspace": 0.03},
    )
    # Plot (top)
    ax0 = axes[0]
    ax0.axis("off")
    ax0.imshow(
        plots_norm[i],
        cmap=cmap,
        interpolation="bilinear",
        origin="lower",
        vmin=0.0,
        vmax=1.0,
        aspect="equal",
    )
    ax0.set_title(
        rf"f{func_id} i{instance_id}  |  $\ell={ell_i:.2f}$  $\Delta={delta_i:.2f}$  $q={iqr_i:.2f}$",
        fontsize=9,
        loc="left",
        pad=1,
    )
    # Mask (bottom)
    ax1 = axes[1]
    ax1.axis("off")
    ax1.imshow(
        mask[i].astype(float) * 0.8 + 0.1,
        cmap="gray",
        interpolation="nearest",
        origin="lower",
        vmin=0,
        vmax=1,
        aspect="equal",
    )

    fig.tight_layout(pad=0.18)
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.03)
    # plt.close(fig)
    saved.append(out_svg)

print("Saved:", demo_svg)
print("Saved colorbar:", colorbar_svg)
print("Saved eval/mask svgs:")
for p in saved:
    print("  ", p)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Keep styling consistent with earlier cells
try:
    plt.style.use("seaborn-v0_8-white")
except Exception:
    pass

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.transparent": False,
    "savefig.facecolor": "white",
    "axes.titlepad": 6,
    "axes.titlesize": 10,
    "font.size": 9,
    "mathtext.fontset": "stix",
    "font.family": "DejaVu Serif",
})

# --------------------------- Batch export settings --------------------------- #
res = 8           # set to 8/16/32 to match your folder name res_{res}
dim = 10
instance_id = 2
rep = 5
n_show = 20
out_dir = "."  # save next to this notebook

base_path = os.path.join("..", "..", "data", "bbob", f"res_{res}")
cmap = "YlGnBu_r"
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

os.makedirs(out_dir, exist_ok=True)

# Save a standalone colorbar once (same cmap + [0,1])
colorbar_svg = os.path.join(out_dir, f"colorbar_res{res}_norm01.svg")
if not os.path.exists(colorbar_svg):
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig_cb, ax_cb = plt.subplots(figsize=(0.5, 15))
    cbar = fig_cb.colorbar(sm, cax=ax_cb)
    cbar.outline.set_linewidth(0.6)
    ax_cb.tick_params(length=2.0, width=0.6, labelsize=12)
    fig_cb.tight_layout(pad=0.15)
    fig_cb.savefig(colorbar_svg, bbox_inches="tight", pad_inches=0.03)
    plt.close(fig_cb)
print("Colorbar:", colorbar_svg)

# --------------------------- Export per function --------------------------- #
missing = []
saved = []
for f_id in range(1, 25):
    npz_path = os.path.join(base_path, f"f{f_id}_i{instance_id}_dim{dim}_rep{rep}.npz")
    if not os.path.exists(npz_path):
        missing.append(npz_path)
        continue

    with np.load(npz_path) as data:
        plots = data["plots"]  # (k, r, r)
        stats = data.get("stats", None)  # (k, 3) expected: (ell, Delta, q)

    k = int(plots.shape[0])
    n = min(n_show, k)
    if n == 0:
        missing.append(npz_path + " (empty plots)")
        continue

    # Figure: 1 row of plots (tight spacing)
    fig_w = 2.05 * n
    fig_h = 2.25
    fig, axes = plt.subplots(1, n, figsize=(fig_w, fig_h))
    axes = np.atleast_1d(axes)

    for i in range(n):
        ax = axes[i]
        ax.axis("off")
        ax.imshow(
            plots[i],
            cmap=cmap,
            interpolation="bilinear",
            origin="lower",
            vmin=0.0,
            vmax=1.0,
            aspect="equal",
        )
        if stats is not None:
            try:
                ell_i, delta_i, iqr_i = stats[i]
                # ax.set_title(rf"$\\ell={ell_i:.2f}\\,\\Delta={delta_i:.2f}\\,q={iqr_i:.2f}$", fontsize=9, loc="left", pad=1)
            except Exception:
                pass

    # Reduce whitespace between panels and outer margins
    fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.01)

    out_svg = os.path.join(out_dir, f"f{f_id}_i{instance_id}_dim{dim}_rep{rep}_first{n}_plots_res{res}.svg")
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01)
    saved.append(out_svg)

print(f"Saved {len(saved)} figures.")
if missing:
    print(f"Missing {len(missing)} files:")
    for p in missing[:10]:
        print("  ", p)
    if len(missing) > 10:
        print("  ...")